# Phase 1 Preprocessing EDA: Canonical Event Normalization & MicroTensor Extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taofeeqhamzat/edge-aui-model-preparation/blob/explore/pipeline/revision/1/notebooks/01_preprocessing_eda.ipynb)

This exploratory data analysis notebook supports the **Edge-AUI Framework** probabilistic "Slow Brain" (INT8 Gated Recurrent Unit). It systematically inspects and verifies:
- **Layer A $\rightarrow$ Layer B:** Parsing raw AdSERP CSV streams and companion XML trial metadata, normalizing coordinates against observed source viewports (`<window>WxH</window>`) into $[0, 1]$ canvas coordinates.
- **Layer B $\rightarrow$ Layer C:** Segmenting event streams into 500ms sliding windows (250ms stride) to extract 9 continuous features (7 core kinematics + 2 contextual scroll features).
- **ADR-001 Modality Masking:** Generating binary Modality Mask Vector $M \in \{0, 1\}^9$ and concatenated 18-dimensional MicroTensors $\widetilde{X}_t = [X_t \odot M, \; M] \in \mathbb{R}^{18}$.
- **Data Quality Assurance:** Verifying zero unhandled NaNs/Infs, strict $[0, 1]$ bounds, and session sequence length distributions.

## 1. Environment Setup & Dependency Configuration
Bootstraps the runtime environment across Google Colab, Kaggle, or local virtual environments, and resolves `src/` module paths.

In [ ]:
# Safe autoreload configuration
try:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'autoreload')
        ip.run_line_magic('autoreload', '2')
except Exception:
    pass

import os
import sys
import glob
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    print('[Environment] Running in Google Colab.')
    if not os.path.exists('src') and not os.path.exists('../src'):
        !git clone -b explore/pipeline/revision/1 https://github.com/taofeeqhamzat/edge-aui-model-preparation.git
        %cd edge-aui-model-preparation
    else:
        try:
            !git checkout explore/pipeline/revision/1
            !git pull origin explore/pipeline/revision/1
        except Exception:
            pass
    !pip install -q huggingface_hub datasets torch pandas numpy matplotlib seaborn onnx onnxruntime pyyaml pyarrow
elif IN_KAGGLE:
    print('[Environment] Running in Kaggle.')
    !pip install -q huggingface_hub datasets torch pandas numpy matplotlib seaborn pyyaml pyarrow
else:
    print('[Environment] Running in local/virtual environment.')

# Configure sys.path: add both project root and src/ to support direct and packaged imports
for p in ['.', '..', 'src', '../src', './model-preparation', './model-preparation/src']:
    abs_p = os.path.abspath(p)
    if os.path.isdir(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)
        print(f'[Path] Added {abs_p} to sys.path')

## 2. Dataset Resolution & Inode-Safe Consolidation
Resolves the AdSERP corpus across local data repositories or DVC/Hugging Face remote storage.

In [ ]:
import importlib
import data
import preprocessing
from config import load_config

# Hot-reload modules to pick up git pulls without restarting kernel
importlib.reload(data)
importlib.reload(preprocessing)

cfg = load_config('minimal')
print(f'[Config] Loaded pipeline configuration: mode={cfg.mode}, input_dim={cfg.input_dim}')

# Ensure dataset availability
adserp_dir = data.ensure_adserp_dataset()
print(f'[Data] AdSERP dataset root: {adserp_dir}')
assert os.path.isdir(adserp_dir), f'AdSERP directory does not exist: {adserp_dir}'
assert (Path(adserp_dir) / 'mouse-movement-data').is_dir(), f'mouse-movement-data not found in {adserp_dir}'

## 3. Viewport Normalization Audit (Layer A $\rightarrow$ Layer B)
Validates the transformation from raw browser mouse coordinates to canvas-normalized coordinates in $[0, 1]$ relative to observed `<window>WxH</window>` XML metadata.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Parse canonical events from a sample session
sample_session = 'p004-b1-t1.csv'
events = preprocessing.parse_adserp_session(sample_session, raw_dir=adserp_dir)
df_events = pd.DataFrame(events)

print(f'Total parsed events for {sample_session}: {len(df_events)}')
print(f'Observed viewport: {df_events["viewport_w"].iloc[0]}x{df_events["viewport_h"].iloc[0]}')
print(f'Coordinate bounds: x_norm in [{df_events["x_norm"].min():.4f}, {df_events["x_norm"].max():.4f}], ' 
      f'y_norm in [{df_events["y_norm"].min():.4f}, {df_events["y_norm"].max():.4f}]')

# Verify [0, 1] bounds
assert (df_events['x_norm'] >= 0.0).all() and (df_events['x_norm'] <= 1.0).all()
assert (df_events['y_norm'] >= 0.0).all() and (df_events['y_norm'] <= 1.0).all()
print('Bounds check PASSED: All canonical coordinates are strictly within [0, 1].')

# Visual comparison: Raw Coordinates vs Normalized Viewport Canvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pointer_df = df_events[df_events['event_type'].isin(['mousemove', 'mouseover', 'click'])]
axes[0].plot(pointer_df['x_raw'], pointer_df['y_raw'], color='navy', alpha=0.6, marker='o', markersize=3)
axes[0].set_title(f'Raw Pointer Trajectory (Source Viewport {df_events["viewport_w"].iloc[0]:.0f}x{df_events["viewport_h"].iloc[0]:.0f})')
axes[0].set_xlabel('X Position (px)')
axes[0].set_ylabel('Y Position (px)')
axes[0].invert_yaxis()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(pointer_df['x_norm'], pointer_df['y_norm'], color='darkgreen', alpha=0.6, marker='o', markersize=3)
axes[1].set_title('Normalized Canvas Trajectory [0, 1]')
axes[1].set_xlabel('Normalized X')
axes[1].set_ylabel('Normalized Y')
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(1.05, -0.05)  # Inverted Y for browser canvas
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. MicroTensor Extraction & Modality Masking ($2D = 18$)
Extracts 18-dimensional MicroTensors across multiple AdSERP sessions using a 500ms sliding window and 250ms stride. Evaluates the 7 core kinematic features and 2 contextual scroll features alongside the binary Modality Mask Vector $M \in \{0, 1\}^9$.

In [ ]:
# Ingest multiple sessions for population-level EDA
adserp_root = Path(adserp_dir)
session_csvs = sorted(list((adserp_root / 'mouse-movement-data').glob('*.csv')))[:30]
print(f'Extracting MicroTensors across {len(session_csvs)} AdSERP sessions...')

all_tensors = []
session_lengths = []

for csv_file in session_csvs:
    evs = preprocessing.parse_adserp_session(str(csv_file), raw_dir=adserp_dir)
    t_seq = preprocessing.extract_session_microtensors(evs, window_size_ms=500, stride_ms=250)
    if len(t_seq) > 0:
        all_tensors.append(t_seq)
        session_lengths.append(len(t_seq))

tensor_matrix = np.concatenate(all_tensors, axis=0)
print(f'Total 18-dim MicroTensor windows extracted: {tensor_matrix.shape[0]}')
print(f'Output feature matrix shape: {tensor_matrix.shape}')
assert tensor_matrix.shape[1] == 18

# Split into Feature Values (0..8) and Modality Mask (9..17)
feature_cols = preprocessing.FEATURE_NAMES
mask_cols = [f'mask_{name}' for name in feature_cols]

df_features = pd.DataFrame(tensor_matrix[:, :9], columns=feature_cols)
df_masks = pd.DataFrame(tensor_matrix[:, 9:], columns=mask_cols)
df_all = pd.concat([df_features, df_masks], axis=1)
df_features.describe().round(4)

## 5. Kinematic Feature Distribution Histograms
Visualizes empirical distributions for the 7 core kinematic metrics and 2 contextual metrics across the $[0, 1]$ interval. Evaluates distribution skewness, natural sensor inactivity zeros, and boundary behavior.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']

for idx, col in enumerate(feature_cols):
    ax = axes[idx]
    vals = df_features[col]
    sns.histplot(vals, bins=30, kde=True, ax=ax, color=colors[idx], edgecolor='none', alpha=0.7)
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Normalized Value [0, 1]')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-0.02, 1.02)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Empirical Distributions: 9 Kinematic & Contextual Micro-Interaction Features', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 6. Boundary & Saturation Rate Diagnostics
Audits boundary rates to distinguish between:
- **Natural Zeros ($0.0000$):** Legitimate physiological or sensor inactivity (e.g., zero scroll events, smooth linear motion without $>45^\circ$ turning angles, or modality mask deactivation when pointer stream is paused).
- **Artificial Ceiling Saturation ($1.0000$):** Behavioral truncation caused by normalization constants compressing high-intensity movements into the ceiling boundary ($X_t \ge \text{scale}$).

Evaluates both population-wide rates and active-window conditioned rates (windows where the feature's modality mask is active).

In [ ]:
# Systematic boundary and saturation rate calculation
sat_records = []
for col in feature_cols:
    vals = df_features[col]
    mask = df_masks[f'mask_{col}']
    active_vals = vals[mask == 1.0]
    
    sat_records.append({
        'Feature': col,
        'Mask Active %': f'{mask.mean()*100:.1f}%',
        'Frac == 0.0': f'{(vals == 0.0).mean()*100:.2f}%',
        'Frac < 0.01': f'{(vals < 0.01).mean()*100:.2f}%',
        'Frac > 0.99': f'{(vals > 0.99).mean()*100:.2f}%',
        'Frac == 1.0': f'{(vals == 1.0).mean()*100:.2f}%',
        'Active == 1.0': f'{(active_vals == 1.0).mean()*100:.2f}%' if len(active_vals) > 0 else 'N/A',
        'P95 (Norm)': f'{np.percentile(vals, 95):.4f}',
        'P99 (Norm)': f'{np.percentile(vals, 99):.4f}',
        'P99.9 (Norm)': f'{np.percentile(vals, 99.9):.4f}'
    })

df_sat_summary = pd.DataFrame(sat_records)
print('=== FEATURE BOUNDARY & SATURATION RATE DIAGNOSTICS ===')
print(df_sat_summary.to_string(index=False))

# Diagnostic analysis notes
print('\nDiagnostic Insights:')
print('1. Velocity Scaling (meanVelocity, maxVelocity): Symmetric denominator (/ 10.0) yields < 0.5% ceiling saturation.')
print('2. Acceleration & Hesitation: Scaled by 1.0 px/ms^2 and 25 turns, keeping active saturation under ~2.2% and ~1.2%.')
print('3. Dwell Time: High ceiling rate (~37% overall, ~50% active) reflects DOM XPath density (40ms/event accumulation in 500ms window).')
print('4. Natural Floor Zeros: Scroll depth/velocity (58% zero) and hesitation (29% zero in active) naturally indicate no-event periods.')

## 7. Modality Mask Activation Frequencies (ADR-001)
Inspects the activation rates of the Modality Mask Vector $M \in \{0, 1\}^9$. Per ADR-001, the mask strictly represents sensor and modality capability (pointer kinematics, DOM dwell, and viewport scroll) to decouple sensor absence from legitimate user stillness without zero-imputation collinearity.

In [ ]:
mask_activation_rates = df_masks.mean()

plt.figure(figsize=(12, 5))
bar_plot = sns.barplot(x=mask_activation_rates.index, y=mask_activation_rates.values, hue=mask_activation_rates.index, palette='Blues_d', legend=False)
plt.title('Modality Mask Activation Frequency ($M \in \{0, 1\}^9$)', fontsize=13, fontweight='bold')
plt.ylabel('Activation Frequency (Fraction of Windows)')
plt.xlabel('Modality Mask Dimensions')
plt.xticks(rotation=40, ha='right')
plt.ylim(0, 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for p in bar_plot.patches:
    bar_plot.annotate(f'{p.get_height():.2f}', 
                      (p.get_x() + p.get_width() / 2., p.get_height()), 
                      ha='center', va='center', xytext=(0, 7), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Data Quality & Sequence Length Integrity Diagnostics
Automated assertions ensuring zero NaNs, zero Infs, bounded tensors, and healthy window count distribution for recurrent GRU sequence modeling.

In [ ]:
# 1. Zero NaN and Inf check
nan_count = int(np.isnan(tensor_matrix).sum())
inf_count = int(np.isinf(tensor_matrix).sum())
assert nan_count == 0, f'Found {nan_count} NaNs in MicroTensor matrix!'
assert inf_count == 0, f'Found {inf_count} Infs in MicroTensor matrix!'
print(f'✅ Integrity Check PASSED: Zero NaNs ({nan_count}) and zero Infs ({inf_count}).')

# 2. Strict [0, 1] bounds verification
min_val = float(tensor_matrix.min())
max_val = float(tensor_matrix.max())
assert min_val >= 0.0, f'Values under 0.0 observed: {min_val}'
assert max_val <= 1.0, f'Values over 1.0 observed: {max_val}'
print(f'✅ Bounds Check PASSED: All {tensor_matrix.size:,} tensor elements bounded in [{min_val:.4f}, {max_val:.4f}].')

# 3. Sequence Length Distribution
plt.figure(figsize=(10, 4))
sns.histplot(session_lengths, bins=15, color='teal', edgecolor='black')
plt.title('Distribution of Windows per AdSERP Session', fontsize=13)
plt.xlabel('Number of 500ms Windows (250ms stride)')
plt.ylabel('Session Count')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Session Length Stats: Min={min(session_lengths)}, Max={max(session_lengths)}, Mean={np.mean(session_lengths):.1f} windows.')

## 9. Summary & Transition to Phase 3

### Findings:
1. **Canonical Viewport Normalization:** Successfully projects diverse source screen resolutions into canonical canvas coordinates in $[0, 1]$, preserving trajectory geometry.
2. **Vectorization & Boundary Dynamics:** MicroTensor extraction produces strictly bounded $[0, 1]$ features. Calibrated normalization scales maintain kinematic ceiling saturation under $2.2\%$, while floor zeros accurately capture true sensor inactivity. Dwell time ceiling rates reflect dense XPath DOM structure interactions.
3. **Modality Masking:** Decouples sensor absence from user inactivity per ADR-001.
4. **Next Step (Phase 3):** Ground MicroTensor sequences in future downstream interaction outcomes over a lookahead horizon $\Delta t \in [500\text{ms}, 1500\text{ms}]$ (`02_target_distribution.ipynb`).